In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [22]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx]
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [25]:
decoded_preds = decode_preds(preds)
decoded_preds[0]

[('sofa',
  tensor(-0.0129, grad_fn=<MulBackward0>),
  1.750870704650879,
  18.473909378051758,
  -29.716907501220703,
  -22.32680320739746),
 ('sofa',
  tensor(-0.0235, grad_fn=<MulBackward0>),
  -4.334051609039307,
  -1.8245720863342285,
  6.819962024688721,
  16.386568069458008),
 ('bottle',
  tensor(-0.0469, grad_fn=<MulBackward0>),
  43.2929801940918,
  -1.4804959297180176,
  15.95975399017334,
  6.467578887939453),
 ('bottle',
  tensor(0.0839, grad_fn=<MulBackward0>),
  87.51888275146484,
  11.91838264465332,
  22.139244079589844,
  -2.5304622650146484),
 ('aeroplane',
  tensor(0.1178, grad_fn=<MulBackward0>),
  30.871601104736328,
  23.52977752685547,
  91.87506103515625,
  -31.768814086914062),
 ('aeroplane',
  tensor(0.0234, grad_fn=<MulBackward0>),
  5.302364349365234,
  3.264976739883423,
  102.27723693847656,
  -7.527675628662109),
 ('cow',
  tensor(-0.0548, grad_fn=<MulBackward0>),
  61.09933090209961,
  -36.03534698486328,
  104.182861328125,
  24.484895706176758),
 ('cow